Colab T4 experiment: Qwen2.5-VL-3B-Instruct, zero-shot vs QLoRA on synthetic drawing crops.
Run cell by cell in Colab (each '# %%' is a cell). GPU runtime required.

VERIFY BEFORE TRUSTING: the Unsloth API below follows Unsloth's public vision
fine-tuning notebooks (FastVisionModel, UnslothVisionDataCollator) as of 2025–26.
If a call errors, check https://unsloth.ai/docs for the current signature.

## 0. helper scripts (embedded so the notebook is self-contained)

In [ ]:
%%writefile gen_crops.py
"""
Synthetic engineering-drawing annotation crops with exact JSON labels.
Generates (1) dimension callouts and (2) GD&T feature control frames.
Output: out/images/*.png and out/train.jsonl, out/test.jsonl

Usage: python gen_crops.py --n 600 --out out
"""
import argparse, json, os, random, sys
from PIL import Image, ImageDraw, ImageFont, ImageFilter

GDT = {  # characteristic -> unicode symbol, needs_datums
    "flatness": ("\u23e5", False),
    "straightness": ("\u23e4", False),
    "circularity": ("\u25cb", False),
    "cylindricity": ("\u232d", False),
    "perpendicularity": ("\u27c2", True),
    "parallelism": ("\u2225", True),
    "angularity": ("\u2220", True),
    "position": ("\u2316", True),
    "concentricity": ("\u25ce", True),
    "symmetry": ("\u232f", True),
    "profile_of_a_surface": ("\u2313", True),
    "profile_of_a_line": ("\u2312", True),
    "circular_runout": ("\u2197", True),
    "total_runout": ("\u2330", True),
}
DIA = "\u2300"


_FONT_WARNED = False


def load_font(size):
    # DejaVu has the GD&T glyphs; fall back to default if missing
    global _FONT_WARNED
    for p in ["/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",        # Debian/Ubuntu/Colab
              "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf",
              "/opt/homebrew/share/fonts/DejaVuSans.ttf",               # macOS (brew install font-dejavu)
              os.path.expanduser("~/Library/Fonts/DejaVuSans.ttf"),
              "/Library/Fonts/DejaVuSans.ttf",
              "DejaVuSans.ttf"]:
        if os.path.exists(p):
            return ImageFont.truetype(p, size)
    if not _FONT_WARNED:
        print("WARNING: DejaVuSans.ttf not found; GD&T symbols will render as boxes. "
              "Generate on Colab/Linux or install DejaVu.", file=sys.stderr)
        _FONT_WARNED = True
    return ImageFont.load_default()


def degrade(img):
    if random.random() < 0.5:
        img = img.filter(ImageFilter.GaussianBlur(random.uniform(0.3, 1.0)))
    if random.random() < 0.3:
        img = img.rotate(random.choice([90, 180, 270]), expand=True, fillcolor="white")
    if random.random() < 0.3:
        img = img.convert("L").point(lambda p: 255 if p > 160 else 0).convert("RGB")
    if random.random() < 0.15:
        img = Image.eval(img, lambda p: 255 - p)
    return img


def make_dimension():
    kind = random.choice(["linear", "diameter", "radius"])
    nominal = round(random.uniform(2, 250), random.choice([0, 1, 2]))
    tol_style = random.choice(["sym", "asym", "fit", "none"])
    qty = random.choice([1, 1, 1, 2, 4, 6])
    prefix = {"linear": "", "diameter": DIA, "radius": "R"}[kind]
    txt = f"{qty}x " if qty > 1 else ""
    txt += f"{prefix}{nominal:g}"
    label = {"type": kind, "quantity": qty, "nominal": nominal, "unit": "mm",
             "upper": None, "lower": None, "fit": None}
    if tol_style == "sym":
        t = random.choice([0.01, 0.02, 0.05, 0.1, 0.2, 0.5])
        txt += f" \u00b1{t:g}"
        label["upper"], label["lower"] = t, -t
    elif tol_style == "asym":
        u = random.choice([0.0, 0.05, 0.1, 0.2]); l = -random.choice([0.05, 0.1, 0.2, 0.3])
        txt += f" +{u:g}/{l:g}"
        label["upper"], label["lower"] = u, l
    elif tol_style == "fit":
        fit = random.choice(["H7", "h6", "g6", "H8", "k6", "p6"])
        txt += f" {fit}"
        label["fit"] = fit
    return txt, label, "dimension"


def make_fcf():
    char = random.choice(list(GDT))
    sym, needs_datums = GDT[char]
    zone_dia = char in ("position", "cylindricity", "concentricity") and random.random() < 0.8
    tol = random.choice([0.005, 0.01, 0.02, 0.025, 0.05, 0.1, 0.2])
    mod = random.choice(["", "", "M", "L"]) if needs_datums else ""
    datums = []
    if needs_datums:
        for d in random.sample(list("ABCD"), random.randint(1, 3)):
            dm = random.choice(["", "", "(M)"])
            datums.append(d + dm)
    cells = [sym, f"{DIA if zone_dia else ''}{tol:g}{'(' + mod + ')' if mod else ''}"] + datums
    label = {"characteristic": char, "symbol": sym, "tolerance": tol,
             "zone": "diameter" if zone_dia else "width",
             "material_modifier": mod or None, "datums": datums}
    return cells, label, "gdt"


def render_text(txt, font):
    pad = 14
    w, h = font.getbbox(txt)[2:]
    img = Image.new("RGB", (w + 2 * pad, h + 2 * pad), "white")
    d = ImageDraw.Draw(img)
    d.text((pad, pad // 2), txt, fill="black", font=font)
    # dimension line underneath, like a drawing
    y = h + pad + 4
    d.line([(4, y), (w + 2 * pad - 4, y)], fill="black", width=1)
    return img


def render_fcf(cells, font):
    pad = 8
    widths = [font.getbbox(c)[2] + 2 * pad for c in cells]
    h = font.getbbox("Ag")[3] + 2 * pad
    img = Image.new("RGB", (sum(widths) + 2, h + 2), "white")
    d = ImageDraw.Draw(img)
    x = 1
    for c, w in zip(cells, widths):
        d.rectangle([x, 1, x + w, h], outline="black", width=2)
        d.text((x + pad, pad // 2), c, fill="black", font=font)
        x += w
    return img


PROMPT_DIM = "Read this dimension callout and return JSON with keys: type, quantity, nominal, unit, upper, lower, fit."
PROMPT_FCF = "Read this GD&T feature control frame and return JSON with keys: characteristic, symbol, tolerance, zone, material_modifier, datums."


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=600)
    ap.add_argument("--out", default="out")
    ap.add_argument("--test_frac", type=float, default=0.17)
    ap.add_argument("--seed", type=int, default=0)
    a = ap.parse_args()
    random.seed(a.seed)
    os.makedirs(os.path.join(a.out, "images"), exist_ok=True)
    rows = []
    seen = set()   # rendered text; a crop string must appear once, so train/test never share one
    attempts = 0
    while len(rows) < a.n and attempts < a.n * 20:
        attempts += 1
        font = load_font(random.choice([22, 26, 30, 34]))
        if random.random() < 0.5:
            txt, label, kind = make_dimension()
            if txt in seen:
                continue
            img = render_text(txt, font)
            prompt = PROMPT_DIM
        else:
            cells, label, kind = make_fcf()
            txt = " | ".join(cells)
            if txt in seen:
                continue
            img = render_fcf(cells, font)
            prompt = PROMPT_FCF
        seen.add(txt)
        img = degrade(img)
        i = len(rows)
        fn = f"images/{kind}_{i:05d}.png"
        img.save(os.path.join(a.out, fn))
        rows.append({"image": fn, "kind": kind, "text": txt, "prompt": prompt,
                     "answer": json.dumps(label, ensure_ascii=False)})
    if len(rows) < a.n:
        print(f"WARNING: only {len(rows)} unique crops possible, asked for {a.n}", file=sys.stderr)
    random.shuffle(rows)
    k = int(len(rows) * a.test_frac)
    for name, part in [("test.jsonl", rows[:k]), ("train.jsonl", rows[k:])]:
        with open(os.path.join(a.out, name), "w", encoding="utf-8") as f:
            for r in part:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"wrote {len(rows) - k} train / {k} test to {a.out}/")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile eval_crops.py
"""
Score model outputs against test.jsonl.
Metrics: JSON-valid rate, exact match per crop, field-level precision/recall/F1,
hallucination rate (= 1 - precision), per kind (dimension / gdt).

Input predictions file: JSONL with {"image": ..., "pred": "<raw model text>"}
Usage: python eval_crops.py --test out/test.jsonl --pred preds_zeroshot.jsonl
"""
import argparse, json, re
from collections import defaultdict


def extract_json(text):
    """Pull the first {...} block out of model text; return dict or None."""
    if text is None:
        return None
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None


def norm(v):
    if v is None:
        return "null"
    if isinstance(v, (int, float)):
        return f"{float(v):g}"
    s = str(v).strip().replace("Ø", "⌀").replace("ø", "⌀")
    try:
        return f"{float(s):g}"
    except ValueError:
        return s.lower()


def flatten(d, prefix=""):
    out = set()
    if isinstance(d, dict):
        for k, v in d.items():
            out |= flatten(v, f"{prefix}{k}.")
    elif isinstance(d, list):
        for i, v in enumerate(d):
            out |= flatten(v, f"{prefix}{i}.")
    else:
        out.add((prefix.rstrip("."), norm(d)))
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--test", required=True)
    ap.add_argument("--pred", required=True)
    a = ap.parse_args()

    gt = {r["image"]: r for r in map(json.loads, open(a.test, encoding="utf-8"))}
    preds = {r["image"]: r["pred"] for r in map(json.loads, open(a.pred, encoding="utf-8"))}

    stats = defaultdict(lambda: {"n": 0, "valid": 0, "em": 0, "tp": 0, "fp": 0, "fn": 0})
    for img, row in gt.items():
        kind = row["kind"]
        for key in (kind, "all"):
            s = stats[key]
            s["n"] += 1
            g = json.loads(row["answer"])
            p = extract_json(preds.get(img))
            if p is None:
                s["fn"] += len(flatten(g))
                continue
            s["valid"] += 1
            G, P = flatten(g), flatten(p)
            if G == P:
                s["em"] += 1
            s["tp"] += len(P & G); s["fp"] += len(P - G); s["fn"] += len(G - P)

    print(f"{'kind':10} {'n':>4} {'json_ok':>8} {'exact':>7} {'prec':>6} {'rec':>6} {'f1':>6} {'halluc':>7}")
    for key in sorted(stats):
        s = stats[key]
        prec = s["tp"] / (s["tp"] + s["fp"]) if s["tp"] + s["fp"] else 0.0
        rec = s["tp"] / (s["tp"] + s["fn"]) if s["tp"] + s["fn"] else 0.0
        f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
        print(f"{key:10} {s['n']:4d} {s['valid']/s['n']:8.2f} {s['em']/s['n']:7.2f} "
              f"{prec:6.2f} {rec:6.2f} {f1:6.2f} {1-prec:7.2f}")


if __name__ == "__main__":
    main()


## 1–2. setup: GPU, install, generate data

In [ ]:
# Colab T4: Runtime -> Change runtime type -> T4 GPU, then run top to bottom.
# Self-contained: the two helper scripts are written by the cells above, no repo clone needed.
!nvidia-smi -L
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes
!python gen_crops.py --n 600 --out out


## 3. load model (4-bit)

In [ ]:
import json, torch
from unsloth import FastVisionModel
from PIL import Image
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error()   # hides the per-call "max_new_tokens and max_length both set" notice

MODEL = "unsloth/Qwen2.5-VL-3B-Instruct"   # Unsloth mirror; fall back to "Qwen/Qwen2.5-VL-3B-Instruct"
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL, load_in_4bit=True, use_gradient_checkpointing="unsloth",
)

def load_rows(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")]

train_rows = load_rows("out/train.jsonl")
test_rows = load_rows("out/test.jsonl")

## 4. inference helper + ZERO-SHOT baseline

In [ ]:
def predict(row, max_new_tokens=128):
    img = Image.open("out/" + row["image"]).convert("RGB")
    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": row["prompt"]}]}]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = tokenizer(img, text, add_special_tokens=False, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def run_eval(rows, out_path):
    FastVisionModel.for_inference(model)
    with open(out_path, "w", encoding="utf-8") as f:
        for i, r in enumerate(rows):
            f.write(json.dumps({"image": r["image"], "pred": predict(r)}, ensure_ascii=False) + "\n")
            if i % 20 == 0: print(i, r["image"])
    print("wrote", out_path)

run_eval(test_rows, "preds_zeroshot.jsonl")
!python eval_crops.py --test out/test.jsonl --pred preds_zeroshot.jsonl

## 5. LoRA config  (language layers only first; set finetune_vision_layers=True for the ablation)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", random_state=0,
)

## 6. dataset in Unsloth's conversation format

In [ ]:
def to_conv(r):
    return {"messages": [
        {"role": "user", "content": [{"type": "image", "image": Image.open("out/" + r["image"]).convert("RGB")},
                                     {"type": "text", "text": r["prompt"]}]},
        {"role": "assistant", "content": [{"type": "text", "text": r["answer"]}]},
    ]}
train_ds = [to_conv(r) for r in train_rows]

## 7. train

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

FastVisionModel.for_training(model)
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_ds,
    args=SFTConfig(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=1, learning_rate=2e-4, warmup_steps=10,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5, optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="linear",
        output_dir="outputs", report_to="none",
        remove_unused_columns=False, dataset_text_field="", dataset_kwargs={"skip_prepare_dataset": True},
        max_seq_length=1024,
    ),
)
stats = trainer.train()
print(stats)

## 8. FINE-TUNED eval

In [ ]:
run_eval(test_rows, "preds_lora.jsonl")
!python eval_crops.py --test out/test.jsonl --pred preds_lora.jsonl

## 9. save adapter

In [ ]:
model.save_pretrained("lora_gdt_crops")
tokenizer.save_pretrained("lora_gdt_crops")